# Cops and Robbers Meeting 11/10/25

## Current Progress:

1. Playable Cops and Robbers Generated with hardcoded strategy:

- Below, illustrates a playable game of cops and robbers where the user controls the robber and the system reacts with a cop that attempts to decrease distance at each time step t.

In [ ]:
#LIA  
initially assume {
}

// Assumptions on the Input Signals
always assume {
  // Grid Parameters
  (MinX() < MaxX());
  (MinY() < MaxY());  

  // Robber must stay within Grid
  (Robber.x <= MaxX()) && (Robber.x >= MinX());
  (Robber.y <= MaxY()) && (Robber.y >= MinY());

  // Robber Relative Positioning
  ! (Cop.x < Robber.x && Cop.x > Robber.x);
  ! (Cop.y < Robber.y && Cop.y > Robber.y);

  // Robber Boundary Reaction
  (Robber.x = MaxX()) -> X ((Robber.x = Robber.x) || (Robber.x = Robber.x - 1));
  (Robber.x = MinX()) -> X ((Robber.x = Robber.x) || (Robber.x = Robber.x + 1));
  (Robber.y = MaxY()) -> X ((Robber.y = Robber.y) || (Robber.y = Robber.y - 1));
  (Robber.y = MinY()) -> X ((Robber.y = Robber.y) || (Robber.y = Robber.y + 1));

  // Robber Movement
  X (Robber.x = Robber.x + 1) || X (Robber.x = Robber.x - 1) || X (Robber.x = Robber.x);
  X (Robber.y = Robber.y + 1) || X (Robber.y = Robber.y - 1) || X (Robber.y = Robber.y);
}

// System Guarantees: Output Signals: Cop Behavior
always guarantee {
  // Cop Boundary Reaction
  // (Cop.x = MaxX()) -> F [Cop.x <- Cop.moveL(Cop.x)];
  // (Cop.x = MinX()) -> F [Cop.x <- Cop.moveR(Cop.x)];
  // (Cop.y = MaxY()) -> F [Cop.y <- Cop.moveU(Cop.y)];
  // (Cop.y = MinY()) -> F [Cop.y <- Cop.moveD(Cop.y)];

  // Cop Movement
  (Cop.x < Robber.x) -> [Cop.x <- Cop.moveR(Cop.x)];
  (Cop.x > Robber.x) -> [Cop.x <- Cop.moveL(Cop.x)];
  (Cop.y > Robber.y) -> [Cop.y <- Cop.moveU(Cop.y)];
  (Cop.y < Robber.y) -> [Cop.y <- Cop.moveD(Cop.y)];
}

### Core Issue
This is not our goal...

We're aiming to illustrate that given the constraints of a cops and robbers game, the cop can find a strategy to win.

This does not formally guarantee a win. Nor does it force the system to actually find its own strategy to win.

## Potential Strategies to do this:

### Use the hardcoded strategy to find the correct assumptions.

Trying to find the correct way to specify the assumptions and the guarantees at the same time is like solving for two sides of an equation. We need some sort of ground truth to narrow what we are solving for...

In [ ]:
always guarantee {
  // Cop Boundary Reaction
  // (Cop.x = MaxX()) -> F [Cop.x <- Cop.moveL(Cop.x)];
  // (Cop.x = MinX()) -> F [Cop.x <- Cop.moveR(Cop.x)];
  // (Cop.y = MaxY()) -> F [Cop.y <- Cop.moveU(Cop.y)];
  // (Cop.y = MinY()) -> F [Cop.y <- Cop.moveD(Cop.y)];

  // Cop Movement
  (Cop.x < Robber.x) -> [Cop.x <- Cop.moveR(Cop.x)];
  (Cop.x > Robber.x) -> [Cop.x <- Cop.moveL(Cop.x)];
  (Cop.y > Robber.y) -> [Cop.y <- Cop.moveU(Cop.y)];
  (Cop.y < Robber.y) -> [Cop.y <- Cop.moveD(Cop.y)];

  F (Cop.x = Robber.x)
}

### Ground Truth Guarantee?

Initially, I realized the cop movement in a finite grid (cop-win graph) led to a win in every case. My assumption then was that it would be trivial to add a formal guarantee that eventually the cop would win. *Although upon rereading "Can Reactive Synthesis and Syntax-Guided Synthesis Be Friends?, this may not be the correct way to specify a recursive loop that leads to some win condition...*

Adding a formal win guarantee here leads to a counter strategy. My intuition was that the robber assumptions were not fully modeling the same assumptions I made of the robber in the actual game play. If I knew in the actual game play that the cop would always win, then in theory I could play around with the assumptions enough to eventually match the real game assumptions and not create any counter strategies.

However, as all things in life, this was easier said than done. I could not figure out how to model the assumptions properly. Will made the claim that the guarantees might be too strict and it may be too difficult to model the assumptions exactly as I played them in the actual game.

In [ ]:
initially assume {
  ! (Robber.x = Cop.x && Robber.y = Cop.y);
}

// Assumptions on the Input Signals
always assume {
  (MinX < MaxX);
  (MinY < MaxY);

  // Robber must stay within Grid
  (Robber.x <= MaxX) && (Robber.x >= MinX);
  (Robber.y <= MaxY) && (Robber.y >= MinY);

  // Robber Relative Positioning
  ! (Cop.x < Robber.x && Cop.x > Robber.x);
  ! (Cop.y < Robber.y && Cop.y > Robber.y);

  // Robber Movements
  (Robber.moveL || Robber.moveR || Robber.stayX) && (Robber.moveU || Robber.moveD || Robber.stayY);

  // Robber Movement Exclusivity
  (! (Robber.moveL && Robber.moveR) && ! (Robber.moveU && Robber.moveD));
  (! (Robber.moveL && Robber.stayX) && ! (Robber.moveR && Robber.stayX));
  (! (Robber.moveU && Robber.stayY) && ! (Robber.moveD && Robber.stayY));

  // Robber Movement Effects
  Robber.moveL -> X (Robber.x = Robber.x - 1);
  Robber.moveR -> X (Robber.x = Robber.x + 1);
  Robber.moveU -> X (Robber.y = Robber.y - 1);
  Robber.moveD -> X (Robber.y = Robber.y + 1);
  Robber.stayX -> X (Robber.x = Robber.x);
  Robber.stayY -> X (Robber.y = Robber.y);
}

always guarantee {
  // Cop Boundary Reaction
  // (Cop.x = MaxX()) -> F [Cop.x <- Cop.moveL(Cop.x)];
  // (Cop.x = MinX()) -> F [Cop.x <- Cop.moveR(Cop.x)];
  // (Cop.y = MaxY()) -> F [Cop.y <- Cop.moveU(Cop.y)];
  // (Cop.y = MinY()) -> F [Cop.y <- Cop.moveD(Cop.y)];

  // Cop Movement
  (Cop.x < Robber.x) -> [Cop.x <- Cop.moveR(Cop.x)];
  (Cop.x > Robber.x) -> [Cop.x <- Cop.moveL(Cop.x)];
  (Cop.y > Robber.y) -> [Cop.y <- Cop.moveU(Cop.y)];
  (Cop.y < Robber.y) -> [Cop.y <- Cop.moveD(Cop.y)];

  F (Cop.x = Robber.x)
}

### Ignore Ground Truth and Mimic Robber Behavior?

As shown above, the robber movement is modeled far more like the actual game. It moves like a king on a king's graph (a cop-win graph).
Maybe we could model our cop by symmetry as they both are king's.

In [ ]:
always guarantee {
  // Cop Boundary Reaction
  (Cop.x >= MinX) && (Cop.x <= MaxX);
  (Cop.y >= MinY) && (Cop.y <= MaxY);

  // Cop Movement
  (Cop.x < Robber.x) -> X (Cop.moveR );
  (Cop.x > Robber.x) -> X (Cop.moveL );
  (Cop.y < Robber.y) -> X (Cop.moveD );
  (Cop.y > Robber.y) -> X (Cop.moveU );

  (Cop.moveL || Cop.moveR || Cop.stayX) && (Cop.moveU || Cop.moveD || Cop.stayY);

  (! (Cop.moveL && Cop.moveR) && ! (Cop.moveU && Cop.moveD));
  (! (Cop.moveL && Cop.stayX) && ! (Cop.moveR && Cop.stayX));
  (! (Cop.moveU && Cop.stayY) && ! (Cop.moveD && Cop.stayY));

  // Cop Movement Effects
  Cop.moveL -> X [Cop.x <- Cop.x - 1];
  Cop.moveR -> X [Cop.x <- Cop.x + 1];
  Cop.moveU -> X [Cop.y <- Cop.y - 1];
  Cop.moveD -> X [Cop.y <- Cop.y + 1];
  Cop.stayX -> X [Cop.x <- Cop.x];
  Cop.stayY -> X [Cop.y <- Cop.y];

  // Cop Capture Condition
  F ((Cop.x = Robber.x) && (Cop.y = Robber.y));
}

### Halting Problem

Synthesis would run up to 30 minutes without any signs of life. I debugged TSLTools, adding a verbose flag to LTLSynt, and allowed TSLTools to output live debugging to terminal. I got:

```
trying to create strategy directly for (!(p0p0eq02robber29x02cop29x & p0p0eq02robber29y02cop29y) & G((p0p0eq02robber29y02cop29y & u02cop29y0f1dadd02cop29y0f1dint241b1b) -> Xp0p0gt02cop29y02robber29y) & G((((p0p0eq02robber29y02cop29y & u02cop29y0f1dadd02cop29y0f1dint241b1b) U p0p0gt02cop29y02robber29y) | G(p0p0eq02robber29y02cop29y & u02cop29y0f1dadd02cop29y0f1dint241b1b)) -> Fp0p0gt02cop29y02robber29y) & G((p0p0eq02robber29y02cop29y & u02cop29y0f1dsub02cop29y0f1dint241b1b) -> Xp0p0lt02cop29y02robber29y) & G((((p0p0eq02robber29y02cop29y & u02cop29y0f1dsub02cop29y0f1dint241b1b) U p0p0lt02cop29y02robber29y) | G(p0p0eq02robber29y02cop29y & u02cop29y0f1dsub02cop29y0f1dint241b1b)) -> Fp0p0lt02cop29y02robber29y) & G((p0p0eq02robber29x02cop29x & u02cop29x0f1dadd02cop29x0f1dint241b1b) -> Xp0p0gt02cop29x02robber29x) & G((((p0p0eq02robber29x02cop29x & u02cop29x0f1dadd02cop29x0f1dint241b1b) U p0p0gt02cop29x02robber29x) | G(p0p0eq02robber29x02cop29x & u02cop29x0f1dadd02cop29x0f1dint241b1b)) -> Fp0p0gt02cop29x02robber29x) & G((p0p0eq02robber29x02cop29x & u02cop29x0f1dsub02cop29x0f1dint241b1b) -> Xp0p0lt02cop29x02robber29x) & G((((p0p0eq02robber29x02cop29x & u02cop29x0f1dsub02cop29x0f1dint241b1b) U p0p0lt02cop29x02robber29x) | G(p0p0eq02robber29x02cop29x & u02cop29x0f1dsub02cop29x0f1dint241b1b)) -> Fp0p0lt02cop29x02robber29x) & G((((p0p0gt02cop29x02robber29x & u02cop29x0f1dsub02cop29x0f1dint241b1b) U p0p0eq02robber29x02cop29x) | G(p0p0gt02cop29x02robber29x & u02cop29x0f1dsub02cop29x0f1dint241b1b)) -> Fp0p0eq02robber29x02cop29x) & G((G(p0p0gt02cop29x02robber29x & u02cop29x0f1dsub02cop29x0f1dint241b1b) | ((p0p0gt02cop29x02robber29x & u02cop29x0f1dsub02cop29x0f1dint241b1b) U p0p0lt02cop29x02robber29x)) -> Fp0p0lt02cop29x02robber29x) & G((G(p0p0gt02cop29x02robber29x & u02cop29x0f1dsub02cop29x0f1dint241b1b) | ((p0p0gt02cop29x02robber29x & u02cop29x0f1dsub02cop29x0f1dint241b1b) U p0p0eq02cop29x02robber29x)) -> Fp0p0eq02cop29x02robber29x) & G((((p0p0lt02cop29x02robber29x & u02cop29x0f1dadd02cop29x0f1dint241b1b) U p0p0eq02robber29x02cop29x) | G(p0p0lt02cop29x02robber29x & u02cop29x0f1dadd02cop29x0f1dint241b1b)) -> Fp0p0eq02robber29x02cop29x) & G((G(p0p0lt02cop29x02robber29x & u02cop29x0f1dadd02cop29x0f1dint241b1b) | ((p0p0lt02cop29x02robber29x & u02cop29x0f1dadd02cop29x0f1dint241b1b) U p0p0gt02cop29x02robber29x)) -> Fp0p0gt02cop29x02robber29x) & G((G(p0p0lt02cop29x02robber29x & u02cop29x0f1dadd02cop29x0f1dint241b1b) | ((p0p0lt02cop29x02robber29x & u02cop29x0f1dadd02cop29x0f1dint241b1b) U p0p0eq02cop29x02robber29x)) -> Fp0p0eq02cop29x02robber29x) & G((((p0p0gt02cop29y02robber29y & u02cop29y0f1dsub02cop29y0f1dint241b1b) U p0p0eq02robber29y02cop29y) | G(p0p0gt02cop29y02robber29y & u02cop29y0f1dsub02cop29y0f1dint241b1b)) -> Fp0p0eq02robber29y02cop29y) & G((G(p0p0gt02cop29y02robber29y & u02cop29y0f1dsub02cop29y0f1dint241b1b) | ((p0p0gt02cop29y02robber29y & u02cop29y0f1dsub02cop29y0f1dint241b1b) U p0p0lt02cop29y02robber29y)) -> Fp0p0lt02cop29y02robber29y) & G((G(p0p0gt02cop29y02robber29y & u02cop29y0f1dsub02cop29y0f1dint241b1b) | ((p0p0gt02cop29y02robber29y & u02cop29y0f1dsub02cop29y0f1dint241b1b) U p0p0eq02cop29y02robber29y)) -> Fp0p0eq02cop29y02robber29y) & G((((p0p0lt02cop29y02robber29y & u02cop29y0f1dadd02cop29y0f1dint241b1b) U p0p0eq02robber29y02cop29y) | G(p0p0lt02cop29y02robber29y & u02cop29y0f1dadd02cop29y0f1dint241b1b)) -> Fp0p0eq02robber29y02cop29y) & G((G(p0p0lt02cop29y02robber29y & u02cop29y0f1dadd02cop29y0f1dint241b1b) | ((p0p0lt02cop29y02robber29y & u02cop29y0f1dadd02cop29y0f1dint241b1b) U p0p0gt02cop29y02robber29y)) -> Fp0p0gt02cop29y02robber29y) & G((G(p0p0lt02cop29y02robber29y & u02cop29y0f1dadd02cop29y0f1dint241b1b) | ((p0p0lt02cop29y02robber29y & u02cop29y0f1dadd02cop29y0f1dint241b1b) U p0p0eq02cop29y02robber29y)) -> Fp0p0eq02cop29y02robber29y) & G((p0p0eq02cop29y02robber29y & u02cop29y0f1dadd02cop29y0f1dint241b1b) -> Xp0p0gt02cop29y02robber29y) & G((((p0p0eq02cop29y02robber29y & u02cop29y0f1dadd02cop29y0f1dint241b1b) U p0p0gt02cop29y02robber29y) | G(p0p0eq02cop29y02robber29y & u02cop29y0f1dadd02cop29y0f1dint241b1b)) -> Fp0p0gt02cop29y02robber29y) & G((p0p0eq02cop29y02robber29y & u02cop29y0f1dsub02cop29y0f1dint241b1b) -> Xp0p0lt02cop29y02robber29y) & G((((p0p0eq02cop29y02robber29y & u02cop29y0f1dsub02cop29y0f1dint241b1b) U p0p0lt02cop29y02robber29y) | G(p0p0eq02cop29y02robber29y & u02cop29y0f1dsub02cop29y0f1dint241b1b)) -> Fp0p0lt02cop29y02robber29y) & G((p0p0eq02cop29x02robber29x & u02cop29x0f1dadd02cop29x0f1dint241b1b) -> Xp0p0gt02cop29x02robber29x) & G((((p0p0eq02cop29x02robber29x & u02cop29x0f1dadd02cop29x0f1dint241b1b) U p0p0gt02cop29x02robber29x) | G(p0p0eq02cop29x02robber29x & u02cop29x0f1dadd02cop29x0f1dint241b1b)) -> Fp0p0gt02cop29x02robber29x) & G((p0p0eq02cop29x02robber29x & u02cop29x0f1dsub02cop29x0f1dint241b1b) -> Xp0p0lt02cop29x02robber29x) & G((((p0p0eq02cop29x02robber29x & u02cop29x0f1dsub02cop29x0f1dint241b1b) U p0p0lt02cop29x02robber29x) | G(p0p0eq02cop29x02robber29x & u02cop29x0f1dsub02cop29x0f1dint241b1b)) -> Fp0p0lt02cop29x02robber29x) & G!(p0p0gt02cop29x02robber29x & p0p0lt02cop29x02robber29x) & G!(p0p0gt02cop29y02robber29y & p0p0lt02cop29y02robber29y)) -> (G((u02cop29y02cop29y & !u02cop29y0f1dadd02cop29y0f1dint241b1b & !u02cop29y0f1dsub02cop29y0f1dint241b1b) | (!u02cop29y02cop29y & u02cop29y0f1dadd02cop29y0f1dint241b1b & !u02cop29y0f1dsub02cop29y0f1dint241b1b) | (!u02cop29y02cop29y & !u02cop29y0f1dadd02cop29y0f1dint241b1b & u02cop29y0f1dsub02cop29y0f1dint241b1b)) & G((u02cop29x02cop29x & !u02cop29x0f1dadd02cop29x0f1dint241b1b & !u02cop29x0f1dsub02cop29x0f1dint241b1b) | (!u02cop29x02cop29x & u02cop29x0f1dadd02cop29x0f1dint241b1b & !u02cop29x0f1dsub02cop29x0f1dint241b1b) | (!u02cop29x02cop29x & !u02cop29x0f1dadd02cop29x0f1dint241b1b & u02cop29x0f1dsub02cop29x0f1dint241b1b)) & !(!(p0p0eq02robber29x02cop29x & p0p0eq02robber29y02cop29y) & G((p0p0eq02robber29y02cop29y & u02cop29y0f1dadd02cop29y0f1dint241b1b) -> Xp0p0gt02cop29y02robber29y) & G((((p0p0eq02robber29y02cop29y & u02cop29y0f1dadd02cop29y0f1dint241b1b) U p0p0gt02cop29y02robber29y) | G(p0p0eq02robber29y02cop29y & u02cop29y0f1dadd02cop29y0f1dint241b1b)) -> Fp0p0gt02cop29y02robber29y) & G((p0p0eq02robber29y02cop29y & u02cop29y0f1dsub02cop29y0f1dint241b1b) -> Xp0p0lt02cop29y02robber29y) & G((((p0p0eq02robber29y02cop29y & u02cop29y0f1dsub02cop29y0f1dint241b1b) U p0p0lt02cop29y02robber29y) | G(p0p0eq02robber29y02cop29y & u02cop29y0f1dsub02cop29y0f1dint241b1b)) -> Fp0p0lt02cop29y02robber29y) & G((p0p0eq02robber29x02cop29x & u02cop29x0f1dadd02cop29x0f1dint241b1b) -> Xp0p0gt02cop29x02robber29x) & G((((p0p0eq02robber29x02cop29x & u02cop29x0f1dadd02cop29x0f1dint241b1b) U p0p0gt02cop29x02robber29x) | G(p0p0eq02robber29x02cop29x & u02cop29x0f1dadd02cop29x0f1dint241b1b)) -> Fp0p0gt02cop29x02robber29x) & G((p0p0eq02robber29x02cop29x & u02cop29x0f1dsub02cop29x0f1dint241b1b) -> Xp0p0lt02cop29x02robber29x) & G((((p0p0eq02robber29x02cop29x & u02cop29x0f1dsub02cop29x0f1dint241b1b) U p0p0lt02cop29x02robber29x) | G(p0p0eq02robber29x02cop29x & u02cop29x0f1dsub02cop29x0f1dint241b1b)) -> Fp0p0lt02cop29x02robber29x) & G((((p0p0gt02cop29x02robber29x & u02cop29x0f1dsub02cop29x0f1dint241b1b) U p0p0eq02robber29x02cop29x) | G(p0p0gt02cop29x02robber29x & u02cop29x0f1dsub02cop29x0f1dint241b1b)) -> Fp0p0eq02robber29x02cop29x) & G((G(p0p0gt02cop29x02robber29x & u02cop29x0f1dsub02cop29x0f1dint241b1b) | ((p0p0gt02cop29x02robber29x & u02cop29x0f1dsub02cop29x0f1dint241b1b) U p0p0lt02cop29x02robber29x)) -> Fp0p0lt02cop29x02robber29x) & G((G(p0p0gt02cop29x02robber29x & u02cop29x0f1dsub02cop29x0f1dint241b1b) | ((p0p0gt02cop29x02robber29x & u02cop29x0f1dsub02cop29x0f1dint241b1b) U p0p0eq02cop29x02robber29x)) -> Fp0p0eq02cop29x02robber29x) & G((((p0p0lt02cop29x02robber29x & u02cop29x0f1dadd02cop29x0f1dint241b1b) U p0p0eq02robber29x02cop29x) | G(p0p0lt02cop29x02robber29x & u02cop29x0f1dadd02cop29x0f1dint241b1b)) -> Fp0p0eq02robber29x02cop29x) & G((G(p0p0lt02cop29x02robber29x & u02cop29x0f1dadd02cop29x0f1dint241b1b) | ((p0p0lt02cop29x02robber29x & u02cop29x0f1dadd02cop29x0f1dint241b1b) U p0p0gt02cop29x02robber29x)) -> Fp0p0gt02cop29x02robber29x) & G((G(p0p0lt02cop29x02robber29x & u02cop29x0f1dadd02cop29x0f1dint241b1b) | ((p0p0lt02cop29x02robber29x & u02cop29x0f1dadd02cop29x0f1dint241b1b) U p0p0eq02cop29x02robber29x)) -> Fp0p0eq02cop29x02robber29x) & G((((p0p0gt02cop29y02robber29y & u02cop29y0f1dsub02cop29y0f1dint241b1b) U p0p0eq02robber29y02cop29y) | G(p0p0gt02cop29y02robber29y & u02cop29y0f1dsub02cop29y0f1dint241b1b)) -> Fp0p0eq02robber29y02cop29y) & G((G(p0p0gt02cop29y02robber29y & u02cop29y0f1dsub02cop29y0f1dint241b1b) | ((p0p0gt02cop29y02robber29y & u02cop29y0f1dsub02cop29y0f1dint241b1b) U p0p0lt02cop29y02robber29y)) -> Fp0p0lt02cop29y02robber29y) & G((G(p0p0gt02cop29y02robber29y & u02cop29y0f1dsub02cop29y0f1dint241b1b) | ((p0p0gt02cop29y02robber29y & u02cop29y0f1dsub02cop29y0f1dint241b1b) U p0p0eq02cop29y02robber29y)) -> Fp0p0eq02cop29y02robber29y) & G((((p0p0lt02cop29y02robber29y & u02cop29y0f1dadd02cop29y0f1dint241b1b) U p0p0eq02robber29y02cop29y) | G(p0p0lt02cop29y02robber29y & u02cop29y0f1dadd02cop29y0f1dint241b1b)) -> Fp0p0eq02robber29y02cop29y) & G((G(p0p0lt02cop29y02robber29y & u02cop29y0f1dadd02cop29y0f1dint241b1b) | ((p0p0lt02cop29y02robber29y & u02cop29y0f1dadd02cop29y0f1dint241b1b) U p0p0gt02cop29y02robber29y)) -> Fp0p0gt02cop29y02robber29y) & G((G(p0p0lt02cop29y02robber29y & u02cop29y0f1dadd02cop29y0f1dint241b1b) | ((p0p0lt02cop29y02robber29y & u02cop29y0f1dadd02cop29y0f1dint241b1b) U p0p0eq02cop29y02robber29y)) -> Fp0p0eq02cop29y02robber29y) & G((p0p0eq02cop29y02robber29y & u02cop29y0f1dadd02cop29y0f1dint241b1b) -> Xp0p0gt02cop29y02robber29y) & G((((p0p0eq02cop29y02robber29y & u02cop29y0f1dadd02cop29y0f1dint241b1b) U p0p0gt02cop29y02robber29y) | G(p0p0eq02cop29y02robber29y & u02cop29y0f1dadd02cop29y0f1dint241b1b)) -> Fp0p0gt02cop29y02robber29y) & G((p0p0eq02cop29y02robber29y & u02cop29y0f1dsub02cop29y0f1dint241b1b) -> Xp0p0lt02cop29y02robber29y) & G((((p0p0eq02cop29y02robber29y & u02cop29y0f1dsub02cop29y0f1dint241b1b) U p0p0lt02cop29y02robber29y) | G(p0p0eq02cop29y02robber29y & u02cop29y0f1dsub02cop29y0f1dint241b1b)) -> Fp0p0lt02cop29y02robber29y) & G((p0p0eq02cop29x02robber29x & u02cop29x0f1dadd02cop29x0f1dint241b1b) -> Xp0p0gt02cop29x02robber29x) & G((((p0p0eq02cop29x02robber29x & u02cop29x0f1dadd02cop29x0f1dint241b1b) U p0p0gt02cop29x02robber29x) | G(p0p0eq02cop29x02robber29x & u02cop29x0f1dadd02cop29x0f1dint241b1b)) -> Fp0p0gt02cop29x02robber29x) & G((p0p0eq02cop29x02robber29x & u02cop29x0f1dsub02cop29x0f1dint241b1b) -> Xp0p0lt02cop29x02robber29x) & G((((p0p0eq02cop29x02robber29x & u02cop29x0f1dsub02cop29x0f1dint241b1b) U p0p0lt02cop29x02robber29x) | G(p0p0eq02cop29x02robber29x & u02cop29x0f1dsub02cop29x0f1dint241b1b)) -> Fp0p0lt02cop29x02robber29x) & G!(p0p0gt02cop29x02robber29x & p0p0lt02cop29x02robber29x) & G!(p0p0gt02cop29y02robber29y & p0p0lt02cop29y02robber29y)))
direct strategy might exist but was not found.
```

LTLSynt kept attempting synthesize after, it appeared. My CPU usage for the core was **100%**.

## Simplify Problem

My next approach was to simplify the problem form 2D to 1D. Doing this yielded a counter-strategy. I don't have the code for that exact iteration with the counter. Will got me to re-read the "Can Reactive Synthesis and Syntax-Guided Synthesis Be Friends?" paper that I hadn't picked up in a while.

From there, I tried to implement recursive calls more as the paper illustrated, simplifying the problem to a cop at some defined point and a robber at another defined point.

In [ ]:
#LIA  
initially assume {
  (Cop.x = 3);
}

// Assumptions on the Input Signals
always assume {
  (Robber.x = 0);
}

// System Guarantees: Output Signals: Cop Behavior
always guarantee {
  // Grid Actions
  ((Cop.x > Robber.x) && (([Cop.x <- Cop.x - 1]) W (Cop.x = Robber.x))) -> F (Cop.x = Robber.x);
}

Interestingly, this **STILL DOES NOT WORK**. It synthesizes a poor solution that freezes at the second time-step.

### Going Forward:

Will and I's intuition is that integer arithmetic as per *LIA* is not behaving as expected. There seems to be a struggle in how TSLMT's conversion to TSL represents the idea of a 1D grid and even more complicated, the idea of a 2D grid.

The initial idea of using a ground truth guarantee seemed promising, but I am unsure it will actually work... Maybe I could modify it to use the proper recursion rule with a pre-condition and post-condition. Yet, this was not even working in the small-scale implementation.

Was that not working because the assumptions were being invalidated? The small-scale implementation synthesized but froze up and failed to carry out the desired task. I make an assumption about the cop which is a system player, but the initial cop position is an input to the system. Perhaps, in such limited assumptions, I give the tool too much freedom to guess assumptions that then get violated when we go from TSLMT to TSL?

Overall, I think a big issue is how we model the idea of a finite grid. The finite grid is key in the proof for a cop-win graph. I think we don't make the proper assumptions on how a 2D finite grid works. Going off of my small-scale example, it appears I can nor figure out how to convey a 1D grid. However, if I can figure out how to illustrate a 1D grid, that is certainly a step towards a 2D grid.

In [ ]:
// Realizable
volatile int cop_x = 0;
volatile int cop_y = 0;
volatile int robber_x = 6;
volatile int robber_y = 6;

volatile int new_input_ready = 0;
volatile int player_dx = 0;
volatile int player_dy = 0;


void read_inputs() {
    // Wait until the game sets new input
    while (!new_input_ready) {
        usleep(1000); // sleep 1ms to avoid busy wait
    }

    // Apply the player input to robber position
    robber_x += player_dx;
    robber_y += player_dy;

    printf("Cop: (%d, %d), Robber: (%d, %d)\n", cop_x, cop_y, robber_x, robber_y);

    // Reset flag
    player_dx = 0;
    player_dy = 0;
    new_input_ready = 0;
}

Test TSL Tools Test Specs in TSL Tools VS Issy to determine if Prog counter checks state.

1. Goal to put this in Sam's CAV Paper.
2.  Want to create another spec that is more complicated.
3.  Two stationary robbers that one cops is going back and forth between.
4.  

maybe open an issue on the git for the stationary robber